# Experiment 0: Raw Pixels + Logistic Regression

**Research Question:** How far can raw visual information go without any feature engineering?

Here we are gonna test the baseline performance by using flattened raw image pixels directly fed into a Logistic Regression model.

## Step 0: Google Colab Setup


In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("\nGoogle Drive Mounted successfully!")
except ImportError:
    print("Not running in Google Colab. Skipping Drive mount.")

# Step 1: Imports and Basics

In [ ]:
import os
import cv2
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

LABEL_MAP = {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}
CLASS_NAMES = list(LABEL_MAP.keys())

# Step 2: Load and Preprocess Data
Read the images, convert them to grayscale, resize them to 128x128, and flatten them into a 1D array of raw pixels.

In [ ]:
def load_and_flatten_images(base_path, target_size=(128, 128)):
    X, y = [], []
    if not os.path.exists(base_path):
        print(f"ERROR: The path {base_path} does not exist! Please check your base_dir variable.")
        return np.array(X), np.array(y)
        
    print(f"Loading data from {base_path}...")
    for class_name, label_idx in tqdm(LABEL_MAP.items(), desc="Classes"):
        class_folder = os.path.join(base_path, class_name)
        if not os.path.exists(class_folder):
            continue
            
        for filename in os.listdir(class_folder):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(class_folder, filename)
                # Read as grayscale
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    # Resize to target size
                    img_resized = cv2.resize(img, target_size)
                    # FLATTEN the image (Raw Pixels)
                    X.append(img_resized.flatten())
                    y.append(label_idx)
    return np.array(X), np.array(y)

base_dir = '/content/drive/MyDrive/NeuroScan'

# --- Auto-detect for local execution ---
if 'base_dir' not in locals():
    current_dir = os.getcwd()
    base_dir = current_dir if os.path.exists(os.path.join(current_dir, 'data', 'Training')) else os.path.dirname(current_dir)

train_dir = os.path.join(base_dir, 'data', 'Training')
test_dir = os.path.join(base_dir, 'data', 'Testing')

print("Loading Training Data...")
X_train, y_train = load_and_flatten_images(train_dir)

print("Loading Testing Data...")
X_test, y_test = load_and_flatten_images(test_dir)

if len(X_train) > 0:
    print(f"\nTraining shape: {X_train.shape} (Images, Pixels)")
    print(f"Testing shape: {X_test.shape} (Images, Pixels)")
else:
    print("\nFailed to load images. Please fix the base_dir path above.")

# Step 3: Train the Model
Train a standard Logistic Regression model directly on the raw pixel values.

In [ ]:
print("Training Logistic Regression on Raw Pixels...")
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
print("Training Complete!")

# Step 4: Evaluation
This cell will output the Accuracy, Precision, Recall, and F1-Score.

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print("================ EVALUATION METRICS ================")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print("====================================================")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix: Raw Pixels using Logistic Regression')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()